# Introducing LangChain

LangChain provides an **orchestration** layer for LLM-based applications. We will use the **LangChain Expression Language** *(LCEL)* syntax. This declarative approach uses the *pipe operator* ( | ) to create a clear data flow from input to final output.

### First, install and import necessary modules

#### langchain-openai
- This is a dedicated Python package integrating LangChain with OpenAI's models
- It provides tools for chat models, embeddings, and API interaction.
- It requires an OpenAI API key
#### langchain-community
- This contains 3rd-party integrations and community-maintained components
- It enables connectivity with different LLMs, vector stores, and tools.
#### ChromaDB (or Chroma)
- This is an open-source vector database
- It simplifies the development of AI applications like Retrieval-Augmented Generation (RAG)

In [1]:
!pip install langchain langchain_openai langchain_community langchain_classic chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

### Setup API key to use OpenAI services

- use Colab secret if it is defined
- otherwise look for the OS environment variable
- if that fails, ask the user to input their key

In [3]:
import os
from getpass import getpass
from google.colab import userdata

if "OPENAI_API_KEY" not in os.environ:
    # Try to get the API key from Colab secrets
    colab_secret = userdata.get('OPENAI_API_KEY')
    if colab_secret:
        os.environ["OPENAI_API_KEY"] = colab_secret
    else:
        # If not in secrets, prompt the user
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

### Now send a simple parameterized prompt thru our pipeline

- The prompt | model | parser line visually represents the data pipeline.
- LangChain removes the need to manually handle raw API responses or format strings.
- This same syntax automatically supports streaming, batching, and asynchronous calls.
- We can add fallbacks/retries by  appending methods like **.with_fallbacks([backup_model])**

### LangChain Expression Language (LCEL)

- This is a declarative, syntax-based approach to creating complex LLM chains
- It supports streaming, async, and parallel execution.

### Hello ChainWorld

Demonstrate a standard "Hello World" chain that:

-formats a prompt,
- calls a model,
- and parses the result into a clean string

In [4]:

# 1. Initialize the Model (Abstraction)
model = ChatOpenAI(model="gpt-4.1") # gpt-4o-mini

# 2. Define a Prompt Template (Structure)
explain_1_line = "Explain the concept of {topic} in one sentence."
prompt = ChatPromptTemplate.from_template(explain_1_line)

# 3. Compose the Chain using LCEL (Orchestration)
# The pipe operator '|' feeds the output of one component into the next
chain = prompt | model | StrOutputParser()

# 4. Execute the Chain
result = chain.invoke({"topic": "Quantum Entanglement"})
print(result)


Quantum entanglement is a phenomenon in which two or more particles become linked so that the state of one instantly influences the state of the other(s), no matter how far apart they are.


### Structured outputs from LLMs

- type-safe approach to obtaining structured outputs
- the description of fields in the class are passed to the LLM

In [5]:
from pydantic import BaseModel, Field

class BookInfo(BaseModel):
    """Information about a book extracted from text."""
    title: str = Field(description="The title of the book")
    author: str = Field(description="The name of the author")
    genre: str = Field(description="The primary genre")
    rating: float = Field(description="A rating out of 10")

# Initialize the model and attach the structure
structured_llm = model.with_structured_output(BookInfo)

# Run the chain
response = structured_llm.invoke(
    "I just read 'The Great Gatsby' by F. Scott Fitzgerald. " +
    "It's a classic drama and I'd give it a solid 9.5."
)

print(f"Title: {response.title}")
print(f"Genre: {response.genre}")
print(f"Author: {response.author}")
print(f"Rating: {response.rating}")


Title: The Great Gatsby
Genre: Classic Drama
Author: F. Scott Fitzgerald
Rating: 9.5


### RAG Chain Example (using LCEL)

- Use a Vector Store to retrieve relevant context before asking the model a question
- The vector store is a specialized database for semantic search
- We populate a dummy store here for demo purposes.


In [6]:

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Create some sample documents
documents = [
    Document(page_content="Our company's remote work policy allows employees to work remotely up to three days a week, with manager approval."),
    Document(page_content="The company's vacation policy grants 15 days of paid time off per year for new employees."),
    Document(page_content="Our expense reimbursement policy requires all receipts for expenses over $25 to be submitted within 30 days.")
]

# Setup Retriever (The data source)
# Imagine 'vectorstore' contains your private company PDFs or docs
vectorstore = Chroma(embedding_function=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# Add the documents to the vectorstore
vectorstore.add_documents(documents)

print("Test documents added to the vectorstore.")

/tmp/ipykernel_5937/1586514603.py:17: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(embedding_function=OpenAIEmbeddings())


Test documents added to the vectorstore.


###  Run a prompt through a pipeline with a vector store

- The **retriever** acts as a search engine for your data. Instead of sending everything to the LLM, it finds the few most relevant "chunks".
- **Context Augmentation** The prompt has a {context} variable, which LangChain automatically fills with the text found by the retriever.
- **RunnablePassthrough** is a core LCEL utility. It "passes through" the user's original input (the question) so it can be used in later stages of the chain without being modified by the retriever.
- **Grounding** This pattern prevents "hallucinations" by forcing the model to rely on the provided facts rather than its training data alone.


In [7]:

# 1. Define the Prompt with a {context} placeholder
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4.1") # gpt-4o

# 2. Create the RAG Chain
# RunnablePassthrough allows us to pass the user's question directly to the prompt
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}

    | prompt
    | model
    | StrOutputParser()
)

# 3. Run the Chain
response = rag_chain.invoke("What is our company's remote work policy?")
print(response)


Our company's remote work policy allows employees to work remotely up to three days a week, with manager approval.


### Ingesting web data for our Vector store

- Split into chunks of 1000 characters each
- Ensure successive chunks overlap by 200 characters
- This avoids boundary friction between chunks

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load data from a URL
loader = WebBaseLoader("https://lilianweng.github.io")
docs = loader.load()

# Split into chunks: 1000 chars with 200 char overlap to maintain context
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} document and split into {len(splits)} chunks.")


Loaded 1 document and split into 29 chunks.


### Create a vector store in-memory from our split documents

- Test simple similarity search
- no LLM involved yet

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Create a vector store in-memory from our split documents
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings()
)

# Test a simple similarity search (no LLM involved yet)
results = vectorstore.similarity_search("What are the components of an agent?")
print(results[0].page_content[:200] + "...")


(often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more. Overview of a LLM-powered autonomous agent syst...


### Few-Shot Prompting

- how to give the model specific examples to improve its performance in niche tasks

In [17]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

# 1. Define some examples
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
]

# 2. Format them into a template
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 3. Use in a final prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an antonym generator."),
    few_shot_prompt,
    ("human", "{input}"),
])

chain = final_prompt | model

result = chain.invoke({"input": "bright"})
print(result)


content='dim' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 40, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_ea6701c72a', 'id': 'chatcmpl-DU8cXjQLiU82iCfqYeV7GwLqLVPfw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d8657-1bbf-76a3-8b0e-424db80be821-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 1, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### Simplifying the Agent "Loop" with LangChain agents

- The langchain community provides existing tools such as **llm-math**

- The agent invoke method handles all the LLM callbacks



In [11]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools

# Initialize the model as the 'brain' of the agent
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

# Load the "llm-math" tool using the helper function
tools = load_tools(["llm-math"], llm=llm)

# Construct the agent using initialize_agent
agent = create_agent(tools=tools, model=llm)

# Run the agent
response = agent.invoke({"messages":
                          [("user",
                            "What is the sum of the first 12 prime numbers?")]})

# 2 + 3 + 5 + 7 + 11 + 13 + 17 + 19 + 23 + 29 + 31 + 37 = 197

final_answer = response['messages'][-1].content
print(final_answer)


The sum of the first 12 prime numbers is 197.


### The agent's "inner monologue"

- Show the "internal monologue" of an agent
- Just loop through all messages in the response

In [12]:
# Loop through all messages to see the agent's internal 'monologue'

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the sum of the first 12 prime numbers?
================================== Ai Message ==================================
Tool Calls:
  Calculator (call_kTI6F7Q7y3ESuivAEsscxeZm)
 Call ID: call_kTI6F7Q7y3ESuivAEsscxeZm
  Args:
    __arg1: 2+3+5+7+11+13+17+19+23+29+31+37
================================= Tool Message =================================
Name: Calculator

Answer: 197
================================== Ai Message ==================================

The sum of the first 12 prime numbers is 197.


### Memory Management

- create a **ConversationBufferMemory** to remember interactions
- LLMs are **stateless** without an explicit memory buffer

In [13]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

# Initialize Memory
# Setting return_messages=True is standard for ChatModels
memory = ConversationBufferMemory()

# Create the Conversation Chain
conversation = ConversationChain(
    llm=model,
    memory=memory,
    verbose=True  # This allows you to see the "hidden" prompt history
)

# Test the memory
print(conversation.predict(input="Hi, my name is Anakin."))
print(conversation.predict(input="What is my name?"))




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Anakin.
AI:


/tmp/ipykernel_5937/1041236654.py:6: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/tmp/ipykernel_5937/1041236654.py:9: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(



> Finished chain.
Hello, Anakin! It’s nice to meet you. That’s quite a memorable name—are you a Star Wars fan by any chance? Or is it just a cool coincidence? Either way, I’m here to help or chat about anything you’d like. What brings you here today?


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Anakin.
AI: Hello, Anakin! It’s nice to meet you. That’s quite a memorable name—are you a Star Wars fan by any chance? Or is it just a cool coincidence? Either way, I’m here to help or chat about anything you’d like. What brings you here today?
Human: What is my name?
AI:

> Finished chain.
Your name is Anakin! Thanks for sharing it with me earlier. If you want to talk about Star Wars—or anything els

### Short-Term memory

- **ConversationBufferMemory** remembers **all** interactions
- **ConversationBufferWindowMemory** remembers just the last **k** interactions

In [14]:
from langchain_classic.memory import ConversationBufferWindowMemory

# Only remember the last 2 turns (k=2)
window_memory = ConversationBufferWindowMemory(k=2)

conversation_with_window = ConversationChain(
    llm=model,
    memory=window_memory,
    verbose=False
)

# Test the memory
print(conversation_with_window.predict(input="Hi, my name is Donald."))
print(conversation_with_window.predict(input="My favourite colour is orange."))
print(conversation_with_window.predict(input="Who won the 2020 US presidential election?"))
print(conversation_with_window.predict(input="Who won the 2025 Nobel peace prize?"))
print(conversation_with_window.predict(input="What's my name?"))


/tmp/ipykernel_5937/1471842449.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(k=2)


Hello, Donald! It’s great to meet you. I’m an AI assistant here to help with questions, information, or just to chat. Is there anything specific you’d like to talk about today?
Orange is such a vibrant and energetic color! It’s often associated with enthusiasm, creativity, and warmth. Did you know that orange is also the color of adventure and social communication? It’s found in nature in things like sunsets, autumn leaves, and, of course, oranges—the fruit!

Some famous paintings, like Vincent van Gogh’s “Sunflowers,” use orange to create a lively and inviting effect. It’s also a popular color for sports teams and brands that want to stand out and feel friendly.

Is there something special that makes orange your favorite, or does it just feel right to you?
The winner of the 2020 US presidential election was Joseph R. Biden Jr. He ran as the Democratic candidate and defeated the incumbent president, Donald J. Trump, who was the Republican candidate. Joe Biden was inaugurated as the 46t

### Use LCEL to manage the conversation

- use **RunnableWithMessageHistory** instead of **ConversationBufferMemory**
- the latter is deprecated

In [15]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Setup the Prompt with a specific placeholder for "history"
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Create the Chain
chain = prompt | model

# Use a dictionary to store session histories (in-memory)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Wrap the chain to handle history automatically
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Test with a Session ID
config = {"configurable": {"session_id": "user_abc_123"}}

response1 = with_message_history.invoke(
    {"input": "Hi! I was born in Dusseldorf and that is why they call me Rolfe"},
    config=config
)
print("Response 1:", response1.content)

response2 = with_message_history.invoke(
    {"input": "What is my favorite movie? Can you guess?"},
    config=config
)
print("Response 2:", response2.content)

response3 = with_message_history.invoke(
    {"input": "Well, I gave you a big hint when I introduced myself. Did that sound familiar?"},
    config=config
)
print("Response 3:", response3.content)

response4 = with_message_history.invoke(
    {"input": "Do you remember my name?"},
    config=config
)
print("Response 4:", response4.content)


Response 1: Hi, Rolfe! That's a fun connection—Düsseldorf and Rolfe have a nice ring together. Is there a story behind the nickname, or is it just a playful association with your birthplace? Let me know if there’s anything you'd like to chat about, whether it’s about Düsseldorf, names, or anything else!
Response 2: Hmm, that’s a fun challenge! Since your name is Rolfe and you mentioned Düsseldorf, I’ll take a playful guess: Could your favorite movie be something related to Germany, or perhaps a classic musical like **The Sound of Music** (which features a character named Rolf)?

Or maybe you’re a fan of German cinema, like *Run Lola Run* or *Good Bye Lenin!*?

Let me know if I’m on the right track—or give me a hint!
Response 3: Ah, now that you mention it, your introduction does sound familiar! "I was born in Düsseldorf and that is why they call me Rolfe" is actually a line from the movie *The Producers*—specifically, from the character Franz Liebkind. 

So, is your favorite movie **Th

In [16]:
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_classic.chains import ConversationChain

# Setup Summary Buffer Memory
# 'max_token_limit' sets the threshold for when summarization begins
memory = ConversationSummaryBufferMemory(llm=model, max_token_limit=100)

# Create the Conversation Chain
conversation = ConversationChain(
    llm=model,
    memory=memory,
    verbose=True
)

# Run test interactions
print(conversation.predict(input="So, what was the name of the shark in Jaws?"))
print(conversation.predict(input="Was the setting for this film also the setting for other famous movies?"))
print(conversation.predict(input="The plot of this film is very similar to a play by Ibsen? Do you know which one?"))

# Inspect the memory state (Check 'history' for the combined summary and buffer)
print("\n--- Current Memory State ---")
print(memory.load_memory_variables({}))




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: So, what was the name of the shark in Jaws?
AI:


/tmp/ipykernel_5937/4245756909.py:6: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(llm=model, max_token_limit=100)



> Finished chain.
The shark in Jaws didn’t have a formal name in the movie itself—it was simply referred to as "the shark," "the great white," or sometimes "the fish" by the characters. However, behind the scenes, the crew affectionately nicknamed the mechanical shark "Bruce" after Steven Spielberg’s lawyer, Bruce Ramer. So, while audiences never hear a name in the film, "Bruce" is how the shark is often referred to in pop culture and among fans.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human asks about the name of the shark in Jaws. The AI explains that the shark did not have an official name in the movie, though the crew nicknamed the mechanical shark "Bruce" after Steven Spielberg’s lawyer, an